# Retrieval-Augmented Generation (RAG) with LangChain

A step-by-step lesson. Each step has a short explanation, then a single code cell you run.
By the end you will have a grounded, conversational Q&A system that answers only from your own documents.

**Before you run anything:** copy `.env.example` to `.env` and paste your Groq key into it. Run the cells top to bottom.

## What RAG is, in one picture

```
   Your documents
        |
   [ 1 LOAD ] --> [ 2 SPLIT ] --> [ 3 EMBED ] --> [ 4 STORE in vector DB ]
                                                          |
   User question  ---------------------------->  [ 5 RETRIEVE the most
                                                     relevant chunks ]
                                                          |
                                                          v
                                            [ 6 GENERATE an answer using
                                              ONLY those chunks as context ]
```

The model never answers from memory — it answers from the chunks we hand it.
That is what keeps it grounded and stops hallucination.

## Step 0 — Install the libraries

Run this once. The versions are pinned to the LangChain 0.3.x family so every import below works exactly as written.
(LangChain 1.x moved some imports to a different package.)

In [ ]:
# Run once per environment. Safe to skip if you already did `pip install -r requirements.txt`.
%pip install -q "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" \
    "langchain-groq>=0.2,<0.4" "langchain-community>=0.3,<0.4" \
    "langchain-text-splitters>=0.3,<0.4" "faiss-cpu>=1.8" \
    "tiktoken>=0.7" "pypdf>=4.0" "python-dotenv>=1.0" "sentence-transformers>=3.0"
print("Done. Restart the kernel if this was the first install.")

## Step 0 — Load your API key

We read the key from the `.env` file so it never appears in the notebook.
Create `.env` next to this notebook with one line:

```
GROQ_API_KEY=gsk-...your key...
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in this folder

assert os.environ.get("GROQ_API_KEY"), \
    "No API key found. Create a .env file with GROQ_API_KEY=gsk-..."
print("API key loaded:", os.environ["GROQ_API_KEY"][:7] + "..." )

## Step 0 — Set up the language model

This is the model that will write the final answer. `temperature=0` makes it deterministic
and removes "creative" guessing — important for a grounded system. We use Groq's free `llama-3.3-70b-versatile`.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("LLM ready:", llm.model_name)

## Step 1 — Load the documents

This is the **Load** step. We read every `.txt`, `.md`, and `.pdf` file from the `data/` folder.
LangChain turns each file into one or more `Document` objects (text + metadata such as the source file name).

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader

DATA_DIR = Path("data")

documents = []
for path in sorted(DATA_DIR.glob("*")):
    if path.suffix.lower() in (".txt", ".md"):
        documents.extend(TextLoader(str(path), encoding="utf-8").load())
    elif path.suffix.lower() == ".pdf":
        documents.extend(PyPDFLoader(str(path)).load())

print(f"Loaded {len(documents)} document(s) from {DATA_DIR}/")
for d in documents:
    print(" -", Path(d.metadata["source"]).name, f"({len(d.page_content)} chars)")

## Step 1.1 — Split the documents into chunks

This is the **Split** step. Whole documents are too big to embed and retrieve precisely,
so we cut them into overlapping chunks. Overlap keeps sentences that straddle a boundary
from being split awkwardly.

- `chunk_size` — characters per chunk.
- `chunk_overlap` — characters shared between neighbouring chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
chunks = splitter.split_documents(documents)

print(f"Split {len(documents)} document(s) into {len(chunks)} chunks")
print("\nExample chunk:\n" + "-" * 50)
print(chunks[0].page_content[:300], "...")

## Step 2 — Turn chunks into embeddings

This is the **Embed** step. An embedding is a list of numbers that captures the meaning of a piece of text.
Texts with similar meaning end up close together in this number space — that is how we later find the chunk
that best matches a question.

We use a local Hugging Face model (`all-MiniLM-L6-v2`) — free, no extra API key needed.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Quick demo: embed a single sentence and look at the vector
sample_vector = embeddings.embed_query("What is Fatima Jawad's current CGPA?")
print("Each embedding is a vector of length:", len(sample_vector))
print("First 5 numbers:", [round(x, 4) for x in sample_vector[:5]])

## Step 3 — Store the embeddings in a vector database (FAISS)

This is the **Store** step. FAISS is a fast, local, in-memory vector database — perfect for teaching
because it needs no server or signup. We embed every chunk and save the index to disk so we don't pay
to rebuild it every time.

In [ ]:
from langchain_community.vectorstores import FAISS

# Embeds all chunks (this is the step that calls the OpenAI embeddings API)
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save to disk so the app and future runs can reuse it instantly
vectorstore.save_local("vectorstore")
print(f"Indexed {vectorstore.index.ntotal} chunks and saved to ./vectorstore/")

## Step 4 — Retrieve: find the chunks that match a question

This is the **Retrieve** step. Given a question, the retriever embeds it and returns the `k` closest chunks.
Let's confirm it pulls the right material before we involve the LLM.

### Sample Example (sentence-transformers demo)

In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load model (IMPORTANT FIX)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast, brown fox leaps over a sleepy dog.",
    "Artificial intelligence is transforming the world.",
    "Machine learning models can predict outcomes based on data.",
    "The sky is clear and blue.",
    "I love reading books on sunny days.",
    "The economy is improving after the recession.",
    "Healthcare advancements are improving life expectancy.",
    "Wildlife conservation is crucial for biodiversity.",
    "Technology is advancing at a rapid pace.",
    "Mathematics is the language of the universe.",
    "The government is focusing on renewable energy sources.",
    "Climate change is a pressing global issue.",
    "Blockchain technology is revolutionizing finance.",
    "Mental health awareness is increasing worldwide.",
    "Quantum computing could solve complex problems faster.",
    "Space exploration is expanding human knowledge.",
    "Music therapy has been shown to reduce stress.",
    "Cybersecurity is essential in the digital age.",
    "Education is the key to a better future."
]

sentence_embeddings = model.encode(sentences, normalize_embeddings=True)

query = "what are the benefits of education"
query_embedding = model.encode(query, normalize_embeddings=True)

similarities = cosine_similarity([query_embedding], sentence_embeddings)

most_similar_index = np.argmax(similarities)

print("Query:", query)
print("Most similar sentence:", sentences[most_similar_index])

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

found = retriever.invoke("What is Fatima Jawad's current CGPA and which university is she studying at?")
print(f"Retrieved {len(found)} chunks. Top match:\n" + "-" * 50)
print("Source:", Path(found[0].metadata["source"]).name)
print(found[0].page_content[:300], "...")

In [ ]:
# DEBUG: see every chunk the retriever actually returns
question = "What is Fatima Jawad's current CGPA and which university is she studying at?"
docs = retriever.invoke(question)
print(f"Retrieved {len(docs)} chunks for: {question!r}\n" + "="*60)
for i, d in enumerate(docs, 1):
    print(f"\n[{i}] source: {Path(d.metadata['source']).name}")
    print(d.page_content[:220].strip())

## Step 5 — Write a grounded prompt (the anti-hallucination guard)

The prompt is where we force the model to behave. We tell it: use only the supplied context,
and if the answer isn't there, say so with a fixed phrase. `{context}` is filled automatically
with the retrieved chunks.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are CVBot, an assistant that answers questions about Fatima Jawad "
     "using ONLY the context below.\n"
     "- Use only facts found in the context; never use outside knowledge.\n"
     "- You MAY combine facts from several context chunks to give a complete answer.\n"
     "- If the answer is genuinely not in the context, say: "
     "\"I don't have that information in the provided CV.\"\n"
     "- Be concise and quote details (dates, CGPA, titles) exactly as written.\n\n"
     "Context:\n{context}"),
    ("human", "{input}"),
])

print("Prompt ready.")

## Step 6 — Build the RAG chain

This is the **Generate** step, wired together. Two pieces:

1. `create_stuff_documents_chain` — stuffs the retrieved chunks into `{context}` and asks the LLM.
2. `create_retrieval_chain` — runs the retriever first, then passes its chunks into piece 1.

The result is a single object you call with a question.

In [ ]:
# Robust imports: works on LangChain 0.3.x and 1.x
try:
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
except ModuleNotFoundError:
    from langchain_classic.chains import create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_chain = create_stuff_documents_chain(llm, answer_prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)

result = rag_chain.invoke({"input": "What is Fatima Jawad's current CGPA and which university is she studying at?"})
print(result["answer"])

## Step 7 — Prove it's grounded (ask something NOT in the data)

If RAG is working, an off-topic question should trigger the fallback instead of a made-up answer.

In [ ]:
result = rag_chain.invoke({"input": "Who is Ehsan here?"})
print(result["answer"])

## Recap

You built a full RAG system:

| Step | Stage | Tool |
|------|-------|------|
| 1 | Load | TextLoader, PyPDFLoader |
| 1.1 | Split | RecursiveCharacterTextSplitter |
| 2 | Embed | OpenAIEmbeddings |
| 3 | Store | FAISS |
| 4 | Retrieve | `vectorstore.as_retriever()` |
| 5–7 | Generate | grounded prompt + `create_retrieval_chain` |

**Why it doesn't hallucinate:** a strict prompt that allows only the retrieved context,
`temperature=0`, and a fixed refusal phrase when the answer isn't present.

**Try next:** drop your own PDFs into `data/`, delete the `vectorstore/` folder, and re-run from Step 2.